# 08i — Bayesian Optimisation with BoTorch

Pipeline 4 integration of 08f endpoint energy, 08g structural plausibility and 08h developability proxies. With only two observed candidates, this is an exploratory demonstration, not a statistically mature optimisation campaign. No affinity score is invented and heterogeneous metrics are kept labelled.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
USE_BOTORCH=True
OUTPUT_ROOT=Path("outputs/pipeline_4_ai_08i_bayesian_optimisation"); TABLE_DIR=OUTPUT_ROOT/"tables"; FIGURE_DIR=OUTPUT_ROOT/"figures"
for d in (TABLE_DIR,FIGURE_DIR): d.mkdir(parents=True,exist_ok=True)
P08F=Path("outputs/pipeline_4_ai_08f_endpoint_energy/production_50snap/tables/08f_ProteinMPNN_cross_method_evidence.csv")
P08G=Path("outputs/pipeline_4_ai_08g_esmfold_structural_validation/tables/08g_ESMFold_structural_validation_summary.csv")
P08H=Path("outputs/pipeline_4_ai_08h_solubility_developability/tables/08h_solubility_developability_summary.csv")
for p in (P08F,P08G,P08H):
 if not p.exists(): print("Missing optional input:",p)


## Load and harmonise observed evidence
Energy, pLDDT and developability are not physically commensurate. We standardise only within this observed table for exploratory acquisition-function inputs, preserving the original columns.

In [2]:
frames=[]
if P08F.exists(): frames.append(pd.read_csv(P08F))
if P08G.exists(): frames.append(pd.read_csv(P08G))
if P08H.exists(): frames.append(pd.read_csv(P08H))
if not frames: raise FileNotFoundError("No 08f/08g/08h result tables found")
df=frames[0]
for x in frames[1:]: df=df.merge(x,on="candidate_id",how="outer",suffixes=("","_dup"))
# remove duplicated merge columns
df=df[[c for c in df.columns if not c.endswith("_dup")]].copy()
print(df.columns.tolist()); display(df)


['candidate_id', 'sequence', 'mean_delta_E_endpoint_kcal_mol', 'sd_delta_E_endpoint_kcal_mol', 'AI_integrated_score', 'ESM2_PLL_mean', 'best_ProteinMPNN_score', 'foldx_interaction_energy_kcal_mol', 'best_I_sc', 'structural_rank', 'structural_rank_sum', 'mean_peptide_CA_RMSD_A', 'mean_peptide_CA_RMSF_A', 'n_persistent_contacts_50pct', 'mean_contact_persistence', 'pdb_path', 'n_atoms', 'n_CA', 'mean_pLDDT_Bfactor', 'pLDDT_available', 'reference_CA_RMSD_A', 'plausibility_rank', 'length', 'invalid_residues', 'net_charge_proxy', 'hydrophobic_fraction', 'aromatic_fraction', 'basic_count', 'acidic_count', 'proline_fraction', 'developability_flag', 'risk_flags', 'screen_rank']


,candidate_id,sequence,mean_delta_E_endpoint_kcal_mol,sd_delta_E_endpoint_kcal_mol,AI_integrated_score,ESM2_PLL_mean,best_ProteinMPNN_score,foldx_interaction_energy_kcal_mol,best_I_sc,structural_rank,...,invalid_residues,net_charge_proxy,hydrophobic_fraction,aromatic_fraction,basic_count,acidic_count,proline_fraction,developability_flag,risk_flags,screen_rank
0,MPNN_NEW_01,TGPRNQYRDLP,-46.408438,4.655577,0.601249,-3.172003,1.2473,-11.7407,-54.169,2,...,NaN,1,0.181818,0.090909,2,1,0.181818,NaN,NaN,1
1,MPNN_NEW_05,IGPRHQYRDLP,-53.673748,6.427169,0.508830,-3.174593,1.4079,-13.7478,-56.118,1,...,NaN,2,0.272727,0.090909,3,1,0.181818,NaN,NaN,2


In [3]:
# Exploratory objective: lower endpoint energy, higher pLDDT, lower hydrophobic fraction.
# Missing values remain missing and are not imputed as favourable.
def z(s, higher=True):
 s=pd.to_numeric(s,errors="coerce"); out=pd.Series(np.nan,index=s.index)
 ok=s.notna();
 if ok.sum()>1: out.loc[ok]=(s.loc[ok]-s.loc[ok].mean())/(s.loc[ok].std(ddof=0) or 1)
 return out if higher else -out
for col,higher in [("mean_delta_E_endpoint_kcal_mol",False),("mean_pLDDT_Bfactor",True),("hydrophobic_fraction",False)]:
 if col in df: df[col+"_z"]=z(df[col],higher)
cols=[c for c in ["mean_delta_E_endpoint_kcal_mol_z","mean_pLDDT_Bfactor_z","hydrophobic_fraction_z"] if c in df]
df["observed_composite_proxy"]=df[cols].mean(axis=1,skipna=False) if cols else np.nan
df.to_csv(TABLE_DIR/"08i_observed_integrated_evidence.csv",index=False); display(df[["candidate_id"]+cols+["observed_composite_proxy"]])


,candidate_id,mean_delta_E_endpoint_kcal_mol_z,mean_pLDDT_Bfactor_z,hydrophobic_fraction_z,observed_composite_proxy
0,MPNN_NEW_01,-1.0,1.0,1.0,0.333333
1,MPNN_NEW_05,1.0,-1.0,-1.0,-0.333333


## BoTorch availability and candidate proposal
A Gaussian-process acquisition function needs more observations than the current two-candidate table to be reliable. If BoTorch is installed, the notebook records that it is available; it does not fabricate a new sequence or pretend two points support a validated optimisation model.

In [4]:
try:
 import botorch, torch
 BOTORCH_AVAILABLE=True
 print("BoTorch:",botorch.__version__)
except Exception as exc:
 BOTORCH_AVAILABLE=False; print("BoTorch unavailable:",exc)
proposal=pd.DataFrame([{"status":"not_proposed","reason":"Only two observed candidates; insufficient data for a defensible GP/acquisition proposal","candidate_id":None,"sequence":None}])
proposal.to_csv(TABLE_DIR/"08i_proposed_candidates.csv",index=False)
rank=df[[c for c in ["candidate_id","sequence","observed_composite_proxy"] if c in df]].copy() if "candidate_id" in df else pd.DataFrame()
if not rank.empty: rank["exploratory_rank"]=rank["observed_composite_proxy"].rank(ascending=False,na_option="bottom",method="min").astype(int); rank.to_csv(TABLE_DIR/"08i_exploratory_ranking.csv",index=False); display(rank)
qc=pd.DataFrame([{"check":"08f_08g_08h_inputs_loaded","pass":len(df)>=2},{"check":"no_unsupported_sequence_proposed","pass":proposal.status.iloc[0]=="not_proposed"},{"check":"botorch_status_reported","pass":True}]); qc.to_csv(TABLE_DIR/"08i_QC.csv",index=False); display(qc); print("ALL QC PASSED:",bool(qc['pass'].all()))
(OUTPUT_ROOT/"08i_report.md").write_text("08i integrates existing evidence. With two candidates, no new sequence is proposed; a validated BoTorch optimisation requires a larger observed design set.\n")


BoTorch unavailable: No module named 'botorch'


,candidate_id,sequence,observed_composite_proxy,exploratory_rank
0,MPNN_NEW_01,TGPRNQYRDLP,0.333333,1
1,MPNN_NEW_05,IGPRHQYRDLP,-0.333333,2


,check,pass
0,08f_08g_08h_inputs_loaded,True
1,no_unsupported_sequence_proposed,True
2,botorch_status_reported,True


ALL QC PASSED: True


156